# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and process the FAIR² dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install -q mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json
from pprint import pprint

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Access metadata
meta = dataset.metadata
print(f"{meta.name}: {meta.description}\n")

## 2. Data Overview
Review available record sets, fields, and their `@id`s.

The Croissant schema organizes data into Record Sets, each identified by a unique `@id`. We'll list the available record sets, along with a preview of their fields (columns), all referenced by their `@id` values.

In [ ]:
# List all record sets with their @id and fields
record_sets = dataset.metadata.record_sets
print(f"Number of record sets: {len(record_sets)}\n")
for rs in record_sets:
    print(f"Record set name: {rs.name}")
    print(f"  @id             : {rs.id}")
    print(f"  description     : {rs.description}")
    if rs.fields:
        print("  Fields:")
        for fld in rs.fields:
            print(f"    - {fld.name:30} (@id: {fld.id}, dataType: {fld.data_type})")
    print('-' * 60)

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s identified previously.

**Example:** We'll extract all record sets and show their columns.

_Replace `<record_set_id>` in the code below with the `@id` you wish to explore!_

In [ ]:
# Collect all record set IDs
record_set_ids = [rs.id for rs in dataset.metadata.record_sets]
dataframes = {}

for rs_id in record_set_ids:
    try:
        records = list(dataset.records(record_set=rs_id))
        if records:
            df = pd.DataFrame(records)
            dataframes[rs_id] = df
            print(f"Loaded {len(df)} records from RecordSet @id='{rs_id}'. Columns:")
            print(df.columns.tolist())
            print(f"First 2 rows:\n{df.head(2)}\n")
        else:
            print(f"RecordSet @id='{rs_id}' yielded 0 records.")
    except Exception as e:
        print(f"Could not load record set {rs_id}: {e}")

## 4. Exploratory Data Analysis (EDA)
Let's analyze one of the main data tables. Pick the relevant RecordSet (by `@id`) and relevant fields for numeric analysis. We'll demonstrate with one (update the variables as desired):

In [ ]:
# Select a RecordSet by its @id
# If record sets were listed as, e.g., 'protein_table', set record_set_id accordingly.
# For demonstration we'll try auto-picking the largest (most records) table.
if dataframes:
    record_set_id = max(dataframes, key=lambda k: len(dataframes[k]))
    df = dataframes[record_set_id]
    print(f"Using RecordSet @id={record_set_id} (rows={len(df)}) for EDA.\nColumns: {df.columns.tolist()}")
    # Attempt to auto-pick a numeric field
    numeric_field = None
    for col in df.columns:
        # Try to find a numeric column
        if pd.api.types.is_numeric_dtype(df[col]):
            numeric_field = col
            break
    if numeric_field is None:
        print("No numeric field found. Attempting to convert columns to numeric.")
        for col in df.columns:
            try:
                df[col] = pd.to_numeric(df[col], errors='coerce')
            except Exception:
                continue
        cand = df.select_dtypes(include='number').columns
        if len(cand) > 0:
            numeric_field = cand[0]
    print(f"Selected numeric field: {numeric_field}")
    threshold = df[numeric_field].quantile(0.75)  # Use Q3 as an example threshold
    filtered_df = df[df[numeric_field] > threshold].copy()
    print(f"Filtered to records with {numeric_field} > {threshold:.3f} (n={len(filtered_df)})\n")
    filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(f"First 5 normalized records (field '{numeric_field}'):")
    print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Try to select a group_field (categorical/text)
    group_field = None
    for col in df.columns:
        if pd.api.types.is_string_dtype(df[col]) and df[col].nunique() < 10:
            group_field = col
            break
    if group_field:
        print(f"\nGrouping by '{group_field}' (unique={df[group_field].nunique()}):")
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
        print(grouped_df.head())
else:
    print("No dataframes loaded. Check schema for available record sets.")

## 5. Visualization
Visualize the distribution of the numeric field and, if present, its breakdown by a group/categorical field.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(8,4))
if 'filtered_df' in locals() and numeric_field in filtered_df.columns:
    sns.histplot(filtered_df[numeric_field].dropna(), kde=True, bins=20, color='skyblue')
    plt.title(f'Distribution of {numeric_field} (>threshold)')
    plt.xlabel(numeric_field)
    plt.ylabel('Count')
    plt.tight_layout()
    plt.show()

    if 'group_field' in locals() and group_field is not None:
        plt.figure(figsize=(10,4))
        sns.boxplot(x=group_field, y=numeric_field, data=filtered_df)
        plt.title(f'{numeric_field} by {group_field}')
        plt.xticks(rotation=45)
        plt.tight_layout()
        plt.show()
else:
    print("No filtered dataframe for visualization. Run previous cells first.")

## 6. Conclusion
In this notebook, we've demonstrated how to load and explore the FAIR² dataset schema and records using the `mlcroissant` library. We've summarized data structures, loaded records into DataFrames, performed elementary EDA (such as threshold filtering and normalization), and visualized distributions. For in-depth analysis, consult the Croissant schema fields documentation and extend the workflow per your project needs.